In [ ]:
# ============================================
# LAKEHOUSE PIPELINE — PATH CONFIGURATION
# ============================================
# Set the base_path widget to your Databricks Unity Catalog
# Volume path before running this notebook.
#
# Default: /Volumes/main/lakehouse_pipeline/raw_data
#   Update this default to match your actual catalog name.
#   (Run: SELECT current_catalog() in a SQL notebook to check.)
#
# How to set the value:
#   Interactive  — Edit the widget text box at the top of the notebook.
#   Workflows    — Pass base_path as a job task parameter;
#                  dbutils.widgets picks it up automatically.
#
# See configs/app_config.yaml for full configuration reference.
# ============================================

# Register widget — update the default value to your Volume path
dbutils.widgets.text(
    "base_path",
    "/Volumes/main/lakehouse_pipeline/raw_data",
    "Base Volume Path"
)

BASE_PATH = dbutils.widgets.get("base_path")

print(f"BASE_PATH configured: {BASE_PATH}")

In [ ]:
# ============================================
# SNOWFLAKE CONNECTION CONFIGURATION
# ============================================
# Credentials are read from environment variables ONLY.
# Never put passwords or account URLs directly in notebook cells.
#
# Required environment variables:
#   SNOWFLAKE_ACCOUNT_URL  e.g. xy12345.snowflakecomputing.com
#   SNOWFLAKE_USER         your Snowflake username
#   SNOWFLAKE_PASSWORD     your Snowflake password
#
# Optional (defaults match configs/app_config.yaml):
#   SNOWFLAKE_DATABASE     default: LAKEHOUSE_DB
#   SNOWFLAKE_SCHEMA       default: ANALYTICS
#   SNOWFLAKE_WAREHOUSE    default: LAKEHOUSE_WH
#   SNOWFLAKE_ROLE         default: ACCOUNTADMIN
#
# In Databricks: set via Cluster Environment Variables or Secrets.
# See configs/app_config.yaml for full configuration reference.
# ============================================

import os

snowflakeOptions = {
    "sfURL":       os.environ.get("SNOWFLAKE_ACCOUNT_URL", ""),
    "sfUser":      os.environ.get("SNOWFLAKE_USER", ""),
    "sfPassword":  os.environ.get("SNOWFLAKE_PASSWORD", ""),
    "sfDatabase":  os.environ.get("SNOWFLAKE_DATABASE", "LAKEHOUSE_DB"),
    "sfSchema":    os.environ.get("SNOWFLAKE_SCHEMA",   "ANALYTICS"),
    "sfWarehouse": os.environ.get("SNOWFLAKE_WAREHOUSE","LAKEHOUSE_WH"),
    "sfRole":      os.environ.get("SNOWFLAKE_ROLE",     "ACCOUNTADMIN")
}

# Validate that required secrets are present before proceeding
_missing = [
    k for k, v in snowflakeOptions.items()
    if k in ("sfURL", "sfUser", "sfPassword") and not v
]
if _missing:
    raise EnvironmentError(
        f"Missing required Snowflake environment variables: {_missing}\n"
        "Set SNOWFLAKE_ACCOUNT_URL, SNOWFLAKE_USER, and SNOWFLAKE_PASSWORD "
        "in your environment before running this notebook."
    )

print("Snowflake configuration loaded from environment variables.")
print(f"Connecting to: {snowflakeOptions['sfDatabase']}.{snowflakeOptions['sfSchema']}")

In [ ]:
# ============================================
# READ GOLD DELTA TABLES
# ============================================

revenue_by_country_df = spark.read.format("delta").load(
    f"{BASE_PATH}/gold/revenue_by_country"
)

top_customers_df = spark.read.format("delta").load(
    f"{BASE_PATH}/gold/top_customers"
)

daily_revenue_df = spark.read.format("delta").load(
    f"{BASE_PATH}/gold/daily_revenue"
)

# ============================================
# WRITE TO SNOWFLAKE
# ============================================

revenue_by_country_df.write \
    .format("snowflake") \
    .options(**snowflakeOptions) \
    .option("dbtable", "REVENUE_BY_COUNTRY") \
    .mode("overwrite") \
    .save()

top_customers_df.write \
    .format("snowflake") \
    .options(**snowflakeOptions) \
    .option("dbtable", "TOP_CUSTOMERS") \
    .mode("overwrite") \
    .save()

daily_revenue_df.write \
    .format("snowflake") \
    .options(**snowflakeOptions) \
    .option("dbtable", "DAILY_REVENUE") \
    .mode("overwrite") \
    .save()

print("Gold Tables Loaded Into Snowflake Successfully!")